# Lab 7: What can a sample tell us about a claim?

Use fictional study data to test a claim about average cannabis use-days in a selected policy context and examine the uncertainty and errors involved in statistical decisions.

**Overall question:** In one selected policy context, does the population average differ from 1 cannabis use-day per person during the previous 30 days?

**Learning objectives**
- Define the eligible population and observation unit from linked outcome and policy-window records.
- Interpret a one-sample test and confidence interval against the 1-day reference, distinguishing statistical evidence from practical magnitude.
- Explain Type I and Type II errors, power, and the limits of repeated-decision simulations.


## Section 1. Meet the measure and plan your question

Use-days describe a person at a visit; policy context describes a place during
the measurement period. Keep this reference table handy as you work.

| Term | What it means in this lab |
| --- | --- |
| **Broad cannabis use-days** | The number of distinct days a person used any of eight selected plant-cannabis products during the **30 completed days before assessment**. Values run from **0 to 30**. A day counts once even if several products were used that day. |
| **Policy context** | The supplied policy category for a fictional place during those same 30 days. **Medical** labels a medical-cannabis policy context; **Recreational** labels an adult nonmedical-use policy context. |
| **Stable window** | The records support the **same policy category for all 30 days**. |
| **Available outcome** | A usable recorded number of use-days, including a documented **zero**. A missing value is an unavailable answer, not evidence of nonuse. |


### Choose the group before seeing its results

Use your team's **E7 Question 7 practice plan** to choose **Medical** or
**Recreational**. Set `CHOICE` to
`'medical'` or `'recreational'` before viewing outcomes. Your population is the
fictional roster with an available broad-use outcome and a stable window in that
category, not all young people nationally.

The demo samples **500 people** from the combined eligible Medical and Recreational
groups; you sample **300** from your chosen group. Your group belongs to the demo
population, so these are not independent groups to compare.

### Record what would count as evidence

The **null hypothesis**, **H₀: μ = 1**, says the population mean (μ) is 1 day.
The **alternative**, **Hₐ: μ ≠ 1**, says it differs in either direction: a
**two-sided** question.

Keep **α = .05**, the decision cutoff used with the p-value in Section 4. Separately,
a **0.5-day meaningful difference** is our criterion for judging size. Evidence
against a claim and practical importance are different questions. Both thresholds
are fictional planning choices, not clinical standards.

Run the setup and check `analysis_plan` before results, so an interesting finding
cannot change the question.


In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import sys
sys.path.insert(0, str(Path('../helpers').resolve()))  # course helper modules
from l7_structure_checks import (check_use_summary, check_decision_summary)

DATA = Path('../data/labs_06_07')
VERSION = '0.6.0-rc1'
NOTEBOOK_VERSION = '3.0.0'

outcomes = pd.read_csv(DATA / 'instructional_cannabis.csv')
window_context = pd.read_csv(DATA / 'instructional_policy_window.csv')
assert len(outcomes) == 8564 and outcomes.session_id.eq('ses-07A').all()
OUTCOME = 'instructional_cannabis_days_30d'
STATUS = 'instructional_cannabis_status'
POLICY = 'instructional_policy_window_code'
WINDOW_STATUS = 'instructional_policy_window_status'
COI = 'le_l_coi__addr1__coi__total__national_score'
UNITS = 'use-days per person per 30 completed days'
MU0, DELTA, ALPHA = 1.0, 0.5, 0.05
CHOICE = 'medical'  # Set medical or recreational BEFORE outcome displays.
MENU = {'medical': {'codes': [2], 'label': 'Medical', 'eligible_N': 3260},
        'recreational': {'codes': [1], 'label': 'Recreational', 'eligible_N': 3429}}
assert CHOICE in MENU, 'Choose medical or recreational.'
STUDENT_CODES = MENU[CHOICE]['codes']
CONTEXT_LABEL = MENU[CHOICE]['label']
DEMO_N, STUDENT_N = 500, 300
DEMO_SEED, STUDENT_SEED = 7100, 7200
EXPLORATION_N, EXPLORATION_SEED = 2000, 7400
assert (DEMO_N, STUDENT_N, MU0, DELTA, ALPHA) == (500, 300, 1., .5, .05)
assert (DEMO_SEED, STUDENT_SEED, EXPLORATION_SEED, EXPLORATION_N) == (7100, 7200, 7400, 2000)
analysis_plan = pd.DataFrame([
    dict(role='demo', context='Stable Medical + Recreational', codes='1,2', n=DEMO_N, seed=DEMO_SEED),
    dict(role='your_turn', context='Stable ' + CONTEXT_LABEL, codes=str(STUDENT_CODES[0]), n=STUDENT_N, seed=STUDENT_SEED),
]).assign(outcome=OUTCOME, units=UNITS, session='ses-07A',
          context_rule='same category throughout [assessment minus 30 days, assessment)',
          mu0=MU0, meaningful_difference=DELTA, alpha=ALPHA, alternative='two-sided',
          method='standard-normal reference; sample SD/sqrt(n); no FPC',
          sampling='SRS without replacement', dataset_version=VERSION, realization=0)
pd.set_option('display.precision', 4)
display(analysis_plan[['role', 'context', 'n', 'seed', 'mu0', 'meaningful_difference', 'alpha', 'alternative']])


### Attach the policy window

The supplied cell attaches each person-visit's policy-window record (joins were taught in
Lab 6) and checks that every outcome record is kept exactly once. The
[field dictionary](../data/labs_06_07/DATA_DICTIONARY.md) documents the columns.


In [ ]:
# Supplied: attach the outcome-window policy record to each person-visit (joins were taught in Lab 6).
KEYS = ['participant_id', 'session_id']
roster = outcomes.merge(window_context, on=KEYS, how='left', validate='one_to_one', indicator=True)
assert len(roster) == len(outcomes) and not roster.duplicated(KEYS).any(), 'Keep each outcome record once.'
pd.testing.assert_frame_equal(roster[outcomes.columns].reset_index(drop=True), outcomes.reset_index(drop=True))
assert roster['_merge'].eq('both').all() and roster[WINDOW_STATUS].ne('stable').sum() == 347
roster = roster.drop(columns='_merge').sort_values(KEYS).reset_index(drop=True)


### Decide which records can answer the question

Every one of the **8,564** outcome records has a policy-window record, but **347** of those
windows are not stable.
Category codes are labels, not a numerical scale: **1 Recreational, 2 Medical,
3 Low THC/CBD, and 4 No recreational/medical**. Missing or changing context is never
relabeled as category 4.

### Why use the policy window rather than baseline?

| Baseline category | Recreational | Medical | Low THC/CBD | No recreational/medical |
| --- | ---: | ---: | ---: | ---: |
| Recreational | 2,404 | 0 | 0 | 0 |
| Medical | 1,154 | 2,207 | 0 | 0 |
| Low THC/CBD | 0 | 888 | 745 | 0 |
| No recreational/medical | 0 | 294 | 254 | 271 |

Rows are baseline categories; columns are categories during the 30-day window for the
**8,217 people with stable windows**. For example, **1,154** people were Medical at
baseline but Recreational during the window. A legal category does not measure shops,
personal access or a young person’s permission to purchase.

The next cells keep people with an available broad-use outcome and a stable window in
the chosen category. **Documented zeros stay in the analysis.**


In [ ]:
# Supplied preparation and demo filter; no policy-history reconstruction.
context_documentation = dict(
    policy_field=POLICY, policy_status_field=WINDOW_STATUS,
    window='assessment minus 30 through assessment minus 1, inclusive',
    stable_rule='one category covers all 30 days',
    residence='unchanged residence is a simulation assumption',
    baseline_field='le_l_lawsmj__mjlaws__state_indicator',
    baseline_limit='historical only; not the outcome-window assignment',
)
demo_eligible = roster.loc[
    roster[WINDOW_STATUS].eq('stable') & roster[POLICY].isin([1, 2])
    & roster[STATUS].isin(['observed', 'documented_nonuse']) & roster[OUTCOME].notna()
].copy()
assert len(demo_eligible) == 6689

# Provided chosen-context filter: same rule, narrower policy selection.
eligible = roster.loc[
    roster[WINDOW_STATUS].eq('stable') & roster[POLICY].isin(STUDENT_CODES)
    & roster[STATUS].isin(['observed', 'documented_nonuse']) & roster[OUTCOME].notna()
].copy()
eligible_N = len(eligible)

In [ ]:
# Provided checks and denominator audit; no full-population outcome means are displayed.
assert not eligible.duplicated(KEYS).any(), 'Keep each participant/session once.'
expected_mask = (roster[WINDOW_STATUS].eq('stable') & roster[POLICY].isin(STUDENT_CODES)
    & roster[STATUS].isin(['observed', 'documented_nonuse']) & roster[OUTCOME].notna())
expected_ids = set(map(tuple, roster.loc[expected_mask, KEYS].to_numpy()))
assert set(map(tuple, eligible[KEYS].to_numpy())) == expected_ids, (
    'Check the supplied outcome-window category and usable outcome. Retain zeros; do not require COI.'
)
assert eligible_N == len(eligible) == MENU[CHOICE]['eligible_N'], 'Use the count of eligible rows, not n=300.'
pd.testing.assert_frame_equal(eligible.sort_values(KEYS).reset_index(drop=True),
    roster.loc[expected_mask].reset_index(drop=True), check_like=True)
eligible = eligible.sort_values(KEYS).reset_index(drop=True)

def audit_context(codes, role, n):
    selected = roster.loc[roster[WINDOW_STATUS].eq('stable') & roster[POLICY].isin(codes)]
    usable = selected.loc[selected[STATUS].isin(['observed', 'documented_nonuse']) & selected[OUTCOME].notna()]
    return dict(role=role, stable_context_N=len(selected), unavailable_outcomes=len(selected)-len(usable),
        eligible_N=len(usable), planned_sample_n=n, sampling_fraction=n/len(usable))
context_audit = pd.DataFrame([
    audit_context([1, 2], 'demo', DEMO_N),
    audit_context(STUDENT_CODES, 'your_turn', STUDENT_N),
])
assert context_audit.sampling_fraction.le(.10).all()
display(context_audit)
print('Eligible population size N differs from planned sample size n. Inference samples have not been drawn.')

### What to notice before continuing

Check `context_audit`. Every outcome record has a policy record, yet not everyone is eligible.
Which two requirements must an eligible record meet, and why does eligible N differ from
the planned sample size n?

For Canvas, `analysis_plan` supports **L7-01–02** and `context_audit` supports **L7-03**.


## Section 2. Get to know the outcome before testing a claim

A mean alone cannot show how many people report zero or how much use-days vary.
We first describe an **exploration sample**: a sample used to understand the data
before the planned test. Predict whether the policy categories will show entirely
separate patterns or substantial overlap.

The supplied code draws **2,000 people from all 8,217 people with stable contexts**
(fixed seed 7400), before checking whether use-days are available, so people with
missing outcomes still count as sampled. All four stable categories appear; only
Medical and Recreational are options for your later test. This draw is separate from
the later inference samples; keep the planned seeds.

Run the supplied setup, complete the grouped summary in Your Turn 1, and then read
the plots alongside the counts.


In [ ]:
exploration_pool = roster.loc[roster[WINDOW_STATUS].eq('stable')].sort_values(KEYS).reset_index(drop=True)
assert len(exploration_pool) == 8217
exploration_rng = np.random.default_rng(EXPLORATION_SEED)
exploration_indices = exploration_rng.choice(len(exploration_pool), size=EXPLORATION_N, replace=False)
exploration_sample = exploration_pool.iloc[exploration_indices].copy()
assert len(exploration_sample) == EXPLORATION_N and not exploration_sample.duplicated(KEYS).any()
exploration_membership = exploration_sample[KEYS].reset_index(drop=True).copy()
exploration_membership.insert(0, 'draw_order', np.arange(1, EXPLORATION_N+1))
exploration_plan = dict(seed=EXPLORATION_SEED, pool_N=len(exploration_pool), sample_n=EXPLORATION_N,
    pool='all stable contexts before outcome filtering', dataset_version=VERSION, realization=0,
    method='default_rng; row indices without replacement; pool sorted by participant/session')

context_labels = {1: 'Recreational', 2: 'Medical', 3: 'Low THC/CBD', 4: 'No recreational/medical'}

# Preserve the sampled records before summarizing usable outcomes.
exploration_sample['context'] = exploration_sample[POLICY].map(context_labels)
usable = (exploration_sample[STATUS].isin(['observed', 'documented_nonuse'])
          & exploration_sample[OUTCOME].notna())
exploration_sample['usable_days'] = exploration_sample[OUTCOME].where(usable)
# Preserve missingness in these indicators; False means observed nonuse, not unknown.
exploration_sample['any_use'] = exploration_sample['usable_days'].gt(0).where(usable)
exploration_sample['nonuse_zero'] = exploration_sample['usable_days'].eq(0).where(usable)

# Worked aggregation: input rows are participants/visits; output rows are contexts.
grouped = exploration_sample.groupby('context', sort=False)
mean_by_context = grouped['usable_days'].mean()

### Your Turn 1: who belongs in the denominator?

The demo calculated a mean for each context. Now build three columns that distinguish
**everyone sampled** from **people whose outcome is available**. Complete the three
marked lines in the supplied function; the surrounding code runs it and aligns the
group labels for you.

For each context, return:

- `sampled_n`: everyone drawn, including people with unavailable outcomes.
- `available_n`: people with a usable value in `usable_days`, including zeros.
- `any_use_percent`: the percentage reporting any use among people whose outcome is available.

For a group with **one user, one documented nonuser and one unavailable outcome**,
predict the two counts and the any-use percentage. Why would filling the missing
answer with `False` change the question?

Useful tools: `size()` counts rows, `count()` counts nonmissing values, and `sum()`
or `mean()` of True/False values counts or averages them. Keep context labels attached
so results align. A group with no available outcomes has a missing percentage, not 0%.

**Optional Copilot learning prompt, after your attempt:** “Use a tiny fictional
table with a zero and a missing outcome to explain why a row count and an available
count can differ. Ask me which denominator answers an any-use question before
showing any code. Do not fill in my lab.”

Run the practice check and the real summary. Available counts should not exceed
sampled counts, and missing outcomes remain visible in the audit.


In [ ]:
# This function applies your three expressions to practice and study data.
def summarize_use(table):
    grouped = table.groupby('context', sort=False)
    sampled_n = None        # TODO: number of sampled records per context
    available_n = None      # TODO: number of usable outcomes per context
    any_use_percent = None  # TODO: percent with any use, using the intended denominator
    assert all(value is not None for value in [sampled_n, available_n, any_use_percent]), 'Complete Your Turn 1: the three labeled summaries.'
    return pd.DataFrame(dict(sampled_n=sampled_n, available_n=available_n,
                            any_use_percent=any_use_percent))

check_use_summary(summarize_use)
use_summary = summarize_use(exploration_sample)
sampled_n = use_summary['sampled_n']
available_n = use_summary['available_n']
any_use_percent = use_summary['any_use_percent']


In [ ]:
# Your function has passed the small-example checks; inspect the study summary's structure.
expected_contexts = set(exploration_sample.context.unique())
for result in [sampled_n, available_n, any_use_percent]:
    assert isinstance(result, pd.Series) and result.index.is_unique, 'Keep one labeled value per context.'
    assert set(result.index) == expected_contexts, 'Keep all sampled contexts.'
assert sampled_n.sum() == len(exploration_sample), 'Do not lose sampled records.'
assert available_n.sum() == exploration_sample.usable_days.notna().sum(), 'Count every available outcome, including zero.'
assert available_n.ge(0).all() and available_n.le(sampled_n).all()
assert any_use_percent.dropna().between(0,100).all(), 'Report a percentage on a 0–100 scale.'

exploration_summary = pd.DataFrame(dict(
    sampled_n=sampled_n, available_n=available_n,
    unavailable_n=sampled_n-available_n,
    nonuse_zero_n=grouped['nonuse_zero'].sum().astype(int),
    any_use_percent=any_use_percent, mean_days=mean_by_context,
)).reindex(list(context_labels.values())).rename_axis('context').reset_index()
exploration_values = {label: exploration_sample.loc[exploration_sample.context.eq(label), 'usable_days'].dropna().to_numpy(float)
                      for label in context_labels.values()}
assert exploration_summary.sampled_n.sum() == EXPLORATION_N
assert (exploration_summary.available_n + exploration_summary.unavailable_n).equals(exploration_summary.sampled_n)
display(exploration_summary)
print('One row per context. Means and percentages use available outcomes, including zeros; these are sample descriptions.')


In [ ]:
# Provided plots. Every usable exploration record contributes, including zero days.
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
bars = axes[0].barh(exploration_summary.context, exploration_summary.any_use_percent,
    color='#327a9a', edgecolor='black')
axes[0].bar_label(bars, labels=[f'{v:.1f}%' for v in exploration_summary.any_use_percent], padding=4)
axes[0].set(xlabel='Any use (% of available exploration records)', title='How common is any use?',
    xlim=(0, max(20, exploration_summary.any_use_percent.max()*1.3)))
styles = ['-', '--', '-.', ':']
distribution_rows = []
for (label, vals), style in zip(exploration_values.items(), styles):
    grid = np.arange(31)
    cdf = np.array([(vals <= day).mean()*100 for day in grid])
    axes[1].step(grid, cdf, where='post', linestyle=style, linewidth=2,
        label=f'{label} (available={len(vals):,})')
    distribution_rows.extend(dict(context=label, days=int(day), percent_at_or_below=float(pct))
        for day, pct in zip(grid, cdf))
exploration_distribution = pd.DataFrame(distribution_rows)
# Expand the vertical scale for readability; choose the floor from the smallest zero percentage.
minimum_percent = exploration_distribution.percent_at_or_below.min()
cdf_floor = max(0, min(80, 5*np.floor((minimum_percent-2)/5)))
axes[1].set(xlabel='Use-days in the 30-day window', ylabel='Available exploration records at or below value (%)',
    title=f'Distribution overlap (vertical axis starts at {cdf_floor:.0f}%)', xlim=(-.3, 30.3), ylim=(cdf_floor, 101))
axes[1].legend(loc='lower right', fontsize=8)
axes[1].grid(alpha=.25)
fig.suptitle('Exploration sample: 2,000 stable-context participants · seed 7400 · zeros retained')
plt.tight_layout()
plt.show()

### What to notice in the table and plots

Start with the counts. **Sampled** includes everyone drawn; **available** includes only
those with a usable outcome, including documented zeros. Any-use percentage asks how
common use is; mean days also reflects how often people used.

The right-hand plot is **cumulative**: its height at **0** is the percentage with no use,
and at **5** the percentage with **5 or fewer use-days**. Its vertical scale is expanded
and does not start at zero.

- Name one pattern and its available-outcome denominator. Does a larger any-use percentage necessarily mean a larger average number of days?
- Would a category label alone tell you an individual's use? Would another sample give the same summaries?

### From describing a sample to answering the planned question

The descriptive comparison does not turn the planned question into a Medical–Recreational
comparison: formal two-group inference comes in Module 8, and neither these patterns nor
the fixed-reference test establishes a causal policy effect. The complete population means
stay hidden until after your sample-based conclusion.

Use `exploration_summary` for **L7-03** and the descriptive part of **L7-10**.


## Section 3. How far is our sample mean from 1 day?

We now return to the question recorded at the start: **does the mean in our chosen population differ from 1 use-day per person per 30 days?** The population mean, **μ**, is the average we want to learn about; our sample mean estimates it. The next cells draw the separately planned inference samples.

### Demo: measure the difference in standard errors

As in Lab 6, **estimated SE = sample SD / √n** describes how much a sample mean could vary across samples. To judge whether a difference from 1 is large relative to that uncertainty, divide it by the SE:

**Standardized statistic (z) = (sample mean − 1) / estimated SE**

A positive z places the estimate above 1 and a negative z below 1; z = 2 means the estimate is two estimated standard errors above 1. We compare z with the **standard-normal curve**, a bell-shaped reference centered at zero. Vu and Harrington §4.3 call the statistic *t* but use this normal reference; Module 8 introduces the t distribution.

**About this method:** the planned n=500 and n=300 two-sided analyses passed the dataset's separate checks. No finite-population correction is used because each sample is under 10% of its population. If SE is zero, the calculation is undefined; keep that failure rather than drawing a replacement sample.


In [ ]:
import math
import platform
from statistics import NormalDist
import scipy
from scipy.stats import norm

# Separate streams; each pool is sorted before sampling row indices without replacement.
demo_pool = demo_eligible.sort_values(KEYS).reset_index(drop=True)
student_pool = eligible.sort_values(KEYS).reset_index(drop=True)
demo_indices = np.random.default_rng(DEMO_SEED).choice(len(demo_pool), size=DEMO_N, replace=False)
student_indices = np.random.default_rng(STUDENT_SEED).choice(len(student_pool), size=STUDENT_N, replace=False)
demo_sample_frame = demo_pool.iloc[demo_indices].copy()
sample_frame = student_pool.iloc[student_indices].copy()
demo_sample = demo_sample_frame[OUTCOME].to_numpy(float)
sample = sample_frame[OUTCOME].to_numpy(float)
assert len(demo_sample) == DEMO_N and len(sample) == STUDENT_N
assert not demo_sample_frame.duplicated(KEYS).any() and not sample_frame.duplicated(KEYS).any()

demo_mean = demo_sample.mean()
demo_sd = demo_sample.std(ddof=1)  # sample SD, not complete-population SD
demo_se = demo_sd / np.sqrt(DEMO_N)
assert np.isfinite(demo_se) and demo_se > 0, 'Undefined demo SE: retain the failure; do not redraw.'
demo_z = (demo_mean - MU0) / demo_se
display(pd.DataFrame([dict(role='demo', n=DEMO_N, mean=demo_mean, SD=demo_sd, SE=demo_se, z=demo_z)]))

### Predict, then run

Before running your own calculation, predict the sign of z if your sample mean is **above 1**, **below 1**, or **exactly 1**. In the code, identify the mean, SD and estimated SE, and explain why we divide by the estimated SE of a mean rather than the SD of individual measurements.


In [ ]:
# Provided calculation for the chosen context.
estimate = sample.mean()
sample_sd = sample.std(ddof=1)
se = sample_sd / np.sqrt(len(sample))
assert np.isfinite(se) and se > 0, 'Undefined SE: retain the failure; do not redraw.'
z = (estimate - MU0) / se

In [ ]:
assert np.isfinite(se) and se > 0, 'Undefined SE: retain the failure; do not redraw.'
assert np.isclose(sample_sd, sample.std(ddof=1), rtol=1e-12, atol=1e-12), 'Use sample SD with ddof=1.'
assert np.isclose(se, sample_sd / np.sqrt(len(sample)), rtol=1e-12, atol=1e-12), 'Divide sample SD by sqrt(n).'
assert np.isclose(z, (estimate - MU0) / se, rtol=1e-12, atol=1e-12), 'Standardize the difference from the fixed reference.'
null_model = pd.DataFrame([
    dict(role='demo', context='Medical + Recreational', n=DEMO_N, mean=demo_mean, SD=demo_sd, SE=demo_se, z=demo_z),
    dict(role='your_turn', context=CONTEXT_LABEL, n=STUDENT_N, mean=estimate, SD=sample_sd, SE=se, z=z),
])
display(null_model)

### What to notice

Read your standardized statistic (z) as a distance: the sample mean is this many estimated standard errors above or below 1. Keep the observed sample mean separate from μ, the unknown population mean in H₀.

Use `null_model` for **L7-04**, keeping the demo row and your chosen-context row distinct. Next, we will ask how unusual that distance would be if the population mean really were 1.


## Section 4. How unusual would this result be if the mean were 1?

Imagine that H₀ is true: the population mean is exactly 1 day. Random samples would still produce different sample means. Would a result as far from 1 as ours be common or unusual under that assumption?

The **p-value** is the probability, if H₀ and the approximation hold, of a standardized statistic (z) at least as far from zero as ours, **in either direction**. A smaller p-value means our result is less consistent with the null model. It is not the probability that H₀ is true, the probability that an individual uses cannabis, or evidence that policy caused a difference.

### Demo: connect the shaded area, decision and interval

The graph shades both tails because our question allows a mean above or below 1; in the code, `norm.sf(abs(z))` finds one tail area and multiplying by 2 adds the other. Our recorded rule is **reject H₀ when p < .05** (α = .05). If p is not below .05, we fail to reject H₀: this sample does not provide enough evidence to conclude that the population mean differs from 1. **Failure to reject does not establish equality.**

The 95% confidence interval gives another view of the same decision: with this two-sided method, the test rejects exactly when the interval excludes 1. Use unrounded numbers for decisions.


In [ ]:
cutoff = norm.ppf(1 - ALPHA / 2)
assert np.isclose(cutoff, NormalDist().inv_cdf(1 - ALPHA/2), rtol=0, atol=1e-12)
demo_p = 2 * norm.sf(abs(demo_z))
demo_margin = cutoff * demo_se
demo_lower = demo_mean - demo_margin
demo_upper = demo_mean + demo_margin
display(pd.DataFrame([dict(role='demo', p_value=demo_p, mean=demo_mean,
    lower=demo_lower, upper=demo_upper, reject_H0=bool(demo_p < ALPHA))]))

### Predict, then run

Use the size of your standardized statistic (z) to predict the shaded area. If z is far from zero, should much or little of the null curve lie at least that far away? Why do we include both tails?

After running, check whether your p-value and 95% interval give the same decision about the reference of 1. Keep the test direction you planned before seeing the results.


In [ ]:
# Provided two-sided p-value and compatible 95% mean interval.
p_value = 2 * norm.sf(abs(z))
margin = cutoff * se
lower = estimate - margin
upper = estimate + margin

In [ ]:
# erfc provides a separate standard-normal tail calculation for the check.
assert np.isclose(p_value, math.erfc(abs(z)/np.sqrt(2)), rtol=1e-12, atol=1e-12), 'Include both normal tails.'
assert np.isclose(margin, cutoff*se, rtol=1e-12, atol=1e-12), 'Use the 95% normal cutoff times SE.'
assert np.isclose(lower, estimate-cutoff*se, rtol=1e-12, atol=1e-12), 'Check the lower endpoint.'
assert np.isclose(upper, estimate+cutoff*se, rtol=1e-12, atol=1e-12), 'Check the upper endpoint.'

def test_row(role, context, n, population_N, mean, SD, SE, statistic, p, lo, hi):
    reject = bool(p < ALPHA)
    decision = ('reject below' if mean < MU0 else 'reject above') if reject else 'fail to reject'
    return dict(role=role, context=context, n=n, eligible_N=population_N, mean=mean, SD=SD, SE=SE,
        z=statistic, p_value=p, lower=lo, upper=hi, reject_H0=reject, decision=decision,
        difference=mean-MU0, difference_lower=lo-MU0, difference_upper=hi-MU0, units=UNITS)

test_summary = pd.DataFrame([
    test_row('demo','Medical + Recreational',DEMO_N,len(demo_pool),demo_mean,demo_sd,demo_se,demo_z,demo_p,demo_lower,demo_upper),
    test_row('your_turn',CONTEXT_LABEL,STUDENT_N,len(student_pool),estimate,sample_sd,se,z,p_value,lower,upper),
])
assert np.array_equal(test_summary.reject_H0, (test_summary.upper < MU0) | (test_summary.lower > MU0))
assert test_summary.p_value.between(0,1).all()
display(test_summary[['role','context','n','mean','SE','z','p_value','lower','upper','decision']])
print('Mean and interval units:', UNITS)

limit = max(4,abs(demo_z)+.5,abs(z)+.5)
grid = np.linspace(-limit,limit,2001)
fig,axes = plt.subplots(1,2,figsize=(11,3.6),sharex=True,sharey=True)
for ax,row in zip(axes,test_summary.to_dict('records')):
    ax.plot(grid,norm.pdf(grid),color='black')
    ax.fill_between(grid,0,norm.pdf(grid),where=np.abs(grid)>=abs(row['z']),color='#327a9a',alpha=.5)
    ax.axvline(row['z'],color='black',linestyle='--',label='Observed standardized statistic (z)')
    ax.set(title=f"{row['role']}: {row['context']}\nn={row['n']}, two-sided p={row['p_value']:.4f}",
        xlabel='Standardized statistic (z)',ylabel='Density')
    ax.legend(fontsize=8)
fig.suptitle('Both shaded tails are at least as extreme under the null reference model')
plt.tight_layout()
plt.show()

### What to notice

Explain the shaded area in a sentence that begins, “If the population mean were 1...” Then state the decision from your p-value and confidence interval and explain why they agree. A decision about evidence does not tell us how large a difference is, and failing to reject does not prove the mean equals 1.

Use `test_summary` and the null-tail display for **L7-04 and the evidence component of L7-07**. We cannot classify this decision as correct or mistaken until the later population-truth check.


## Section 5. Could the difference matter?

A small p-value can show evidence of a difference without showing that it is large enough
to matter. Our separate criterion is a difference of **0.5 day per person per 30 days**,
a fictional planning threshold, not a clinical one.

**Estimated difference = sample mean − 1 day**

Subtracting 1 from both interval endpoints puts the confidence interval on the same scale:
a mean interval from 0.8 to 1.3 becomes a difference interval from −0.2 to +0.3. Because the
reference is fixed, subtracting it adds no uncertainty. Read the difference interval against
two landmarks:

- **Zero** means no difference from the reference; whether the interval includes zero matches the test's decision.
- **−0.5 and +0.5 day** mark the meaningful-difference boundaries.

An interval can include zero and also a meaningful departure. An interval entirely between
−0.5 and +0.5 suggests smaller differences under this model, but it is not a formal
equivalence test. The display keeps the mean scale beside the difference scale.


In [ ]:
def magnitude_relation(lo, hi, delta):
    if lo >= delta or hi <= -delta:
        return 'entirely at/beyond a meaningful boundary'
    if lo > -delta and hi < delta:
        return 'entirely within the meaningful band'
    return 'includes a meaningful boundary'

interpretation_table = test_summary[['role','context','difference','difference_lower','difference_upper']].copy()
interpretation_table['meaningful_difference'] = DELTA
interpretation_table['point_departure_at_least_delta'] = interpretation_table.difference.abs().ge(DELTA)
interpretation_table['magnitude_relation'] = [
    magnitude_relation(lo,hi,DELTA) for lo,hi in zip(interpretation_table.difference_lower,interpretation_table.difference_upper)]
display(interpretation_table[['role','context','difference','difference_lower','difference_upper','magnitude_relation']])

fig,axes = plt.subplots(1,2,figsize=(12,4.2))
for i,row in enumerate(test_summary.to_dict('records')):
    axes[0].errorbar(row['mean'],i,xerr=[[row['mean']-row['lower']],[row['upper']-row['mean']]],
        fmt='o',color='black',capsize=5)
    axes[1].errorbar(row['difference'],i,
        xerr=[[row['difference']-row['difference_lower']],[row['difference_upper']-row['difference']]],
        fmt='o',color='black',capsize=5)
labels = [f"Demo n={DEMO_N}",f"{CONTEXT_LABEL} n={STUDENT_N}"]
for ax in axes:
    ax.set(yticks=[0,1],yticklabels=labels,ylim=(-.6,1.6))
    ax.grid(axis='x',alpha=.2)
axes[0].axvline(MU0,color='#327a9a',linestyle='--',label='Fixed reference: 1 day')
axes[0].set(title='95% intervals for each population mean',xlabel='Use-days per person per 30 days')
axes[0].legend(loc='upper center',fontsize=8)
axes[1].axvspan(-DELTA,DELTA,color='#ddebf1',alpha=.65)
axes[1].axvline(0,color='black',label='Zero difference')
for j,value in enumerate([-DELTA,DELTA]):
    axes[1].axvline(value,color='#327a9a',linestyle='--',label='Meaningful boundary: ±0.5' if j==0 else None)
axes[1].set(title='The same intervals, minus the fixed reference',xlabel='Difference from 1 day (original outcome units)')
axes[1].legend(loc='upper center',fontsize=8)
fig.suptitle('Each row answers its own one-mean question; these are not group-comparison intervals')
plt.tight_layout()
plt.show()

### What to notice

State your sample-based answer: name the population, give the estimated mean and its interval, and describe the evidence about the reference of 1. Then read your difference interval: does it include zero, and does it include or extend beyond either ±0.5-day boundary? Comparing the demo and chosen-context p-values, or checking whether their intervals overlap, does not test a difference between those groups.

Use `interpretation_table` and the paired interval display for **L7-09 and the inference component of L7-10**. Record your conclusion before continuing: your population's true mean has not been revealed.

**End-of-class checkpoint:** save your sample-based conclusion now. Resume with Section 6A for homework, due with Section 7 and Canvas before Module 8 begins.


## Section 6A. What happens when we repeat the decision?

Even a correctly applied method makes some mistaken decisions, because samples vary. To see
how often, we need populations whose true means we know. The three practice populations below
are **separate from the Medical/Recreational roster**, whose means are still hidden.

| Practice population | Values among its 10,000 people | True mean |
| --- | --- | --- |
| Null: E7's pattern R | 8,000 zeros and 2,000 fives (20% at 5 days) | 1 day |
| A | 7,000 zeros and 3,000 fives | 1.5 days |
| B | 8,000 zeros, 1,000 sevens and 1,000 eights | 1.5 days |

The supplied code takes 2,000 random samples of **250 people without replacement** from each
and tests each against H₀: μ=1 with the same two-sided method and α=.05. We can then compare
each decision with the known truth:

| Population truth | Reject H₀ | Fail to reject H₀ |
| --- | --- | --- |
| μ=1 (null population) | **Type I error:** conclude there is a difference when the null is true | Correct nonrejection |
| μ≠1 (A or B) | Correct rejection; the probability of this is **power** | **Type II error:** miss a real difference |

`simulation_results` has **one row per population and repetition**. `reject_H0` is True, False,
or missing if SE is undefined; an undefined attempt stays in the table and is never changed to False.

A and B have equal means but different spreads. Must their power be equal? Predict before reading the rates.


In [ ]:
# Provided simulation machinery; each population is ordered by its outcome value.
SIM_N, SIM_REPS = 250, 2000
SIM_SEEDS = {'null': 7300, 'A': 7301, 'B': 7302}
error_populations = {
    'null': np.repeat([0., 5.], [8000, 2000]),
    'A': np.repeat([0., 5.], [7000, 3000]),
    'B': np.repeat([0., 7., 8.], [8000, 1000, 1000]),
}
simulation_population_summary = pd.DataFrame([
    dict(population=name, N=len(values), true_mean=values.mean(), population_SD=values.std(ddof=0),
         sample_n=SIM_N, repetitions=SIM_REPS, seed=SIM_SEEDS[name])
    for name, values in error_populations.items()
])
display(simulation_population_summary)

def run_decision_simulations(populations, seeds, n, repetitions):
    rows = []
    for name, values in populations.items():
        rng = np.random.default_rng(seeds[name])
        for repetition in range(1, repetitions+1):
            indices = rng.choice(len(values), size=n, replace=False)
            draw = values[indices]
            mean = draw.mean()
            sd = draw.std(ddof=1)
            estimated_se = sd / np.sqrt(n)
            defined = bool(np.isfinite(estimated_se) and estimated_se > 0)
            statistic = (mean-MU0)/estimated_se if defined else np.nan
            p = float(2*norm.sf(abs(statistic))) if defined else np.nan
            rows.append(dict(population=name, repetition=repetition, mean=mean, SD=sd,
                SE=estimated_se, z=statistic, p_value=p,
                reject_H0=bool(p < ALPHA) if defined else pd.NA,
                undefined=not defined))
    result = pd.DataFrame(rows)
    result['reject_H0'] = result['reject_H0'].astype('boolean')
    assert not result.duplicated(['population','repetition']).any()
    return result

simulation_results = run_decision_simulations(error_populations, SIM_SEEDS, SIM_N, SIM_REPS)
display(simulation_results.head(3))
print('Simulation row unit: one population/repetition. Total rows:', len(simulation_results))

### Your Turn 2: what counts as a usable decision?

Each row is **one population and one repetition**. As in Your Turn 1, distinguish
attempts from results: here, attempted tests versus tests that produced a decision.
The same repetition number appears in different populations, so keep the populations
separate.

Complete two lines in the supplied function: count the True values, then choose
the denominator for the requested rate. `repetition_counts` includes every attempt;
`defined_counts` includes only True/False decisions. Our question is:

> Among attempts that produced a decision, what fraction rejected H₀?

Predict the rejection count, denominator and rate for **True, False, missing** in one
population. If every decision is missing, should its rate be 0 or missing? Would replacing
the missing value with False answer the same question? The supplied code keeps an
all-missing population in the result with a missing rate; missing attempts stay in the table.

**Optional Copilot learning prompt, after your attempt:** “Give me four fictional
test attempts including True, False and missing decisions. Ask me to explain the
difference between all attempts and usable decisions. Help me reason about the
denominator without completing my code.”

Run the practice check and the real summary. Identify which population's rejection
rate describes Type I error and which describe power.


In [ ]:
def summarize_decisions(table):
    # Group structure is supplied; choose how to count and which denominator to use.
    groups = table.groupby('population')
    decision_groups = groups['reject_H0']
    repetition_counts = groups.size()
    defined_counts = decision_groups.count()
    rejection_counts = None  # TODO: number of True decisions in each population
    rate_denominator = None  # TODO: choose repetition_counts or defined_counts
    assert all(value is not None for value in [rejection_counts, rate_denominator]), 'Complete Your Turn 2: the rejection count and denominator.'
    rejection_rates = rejection_counts / rate_denominator.replace(0, np.nan)
    return pd.DataFrame(dict(rejections=rejection_counts, rejection_rate=rejection_rates))

check_decision_summary(summarize_decisions)
decision_summary = summarize_decisions(simulation_results)
rejection_counts = decision_summary['rejections']
rejection_rates = decision_summary['rejection_rate']


In [ ]:
# Keep the groups separate and keep undefined attempts in the audit.
for result in [rejection_counts, rejection_rates]:
    assert isinstance(result, pd.Series) and result.index.is_unique, 'Return one labeled value per population.'
    assert set(result.index) == set(error_populations), 'Keep null, A and B separate.'
assert rejection_counts.ge(0).all()
assert rejection_rates.dropna().between(0,1).all(), 'Report a proportion on a 0–1 scale.'

audit_groups = simulation_results.groupby('population')
error_rate_sim = pd.DataFrame(dict(
    repetitions=audit_groups.size(), defined_decisions=audit_groups['reject_H0'].count(),
    undefined_runs=audit_groups['undefined'].sum(), rejections=rejection_counts,
    rejection_rate=rejection_rates,
)).reindex(['null','A','B']).rename_axis('population').reset_index()
error_rate_sim['nonrejections'] = error_rate_sim.defined_decisions-error_rate_sim.rejections
assert (error_rate_sim.defined_decisions + error_rate_sim.undefined_runs).eq(SIM_REPS).all()
error_rate_sim['rate_meaning'] = ['Type I error','Power','Power']
error_rate_sim['type_II_rate'] = np.where(error_rate_sim.population.eq('null'), np.nan, 1-error_rate_sim.rejection_rate)
error_rate_sim['MC95_margin'] = 1.96*np.sqrt(
    error_rate_sim.rejection_rate*(1-error_rate_sim.rejection_rate)/error_rate_sim.defined_decisions)
error_rate_sim['denominator_note'] = np.where(error_rate_sim.undefined_runs.eq(0),
    'all repetitions defined', 'conditional on defined decisions; inspect failures')
display(error_rate_sim[['population','repetitions','defined_decisions','undefined_runs','rejections',
    'rejection_rate','MC95_margin','rate_meaning','type_II_rate']])
print('Monte Carlo margins quantify simulation noise in rates; they are not intervals for mean use-days.')
if error_rate_sim.undefined_runs.any():
    print('Undefined decisions retained: rates describe only defined runs. Do not report unconditional error/power.')


In [ ]:
# Provided comparison: the outcome on the vertical axis is a decision rate, not mean days.
fig, ax = plt.subplots(figsize=(8,4))
positions = np.arange(3)
ax.errorbar(positions, error_rate_sim.rejection_rate.to_numpy(float),
    yerr=error_rate_sim.MC95_margin.to_numpy(float), fmt='o', color='black', capsize=6)
ax.set(xticks=positions, xticklabels=['Null: Type I error','A: power','B: power'],
       xlim=(-.5,2.5), ylim=(-.03,1.06), ylabel='Proportion of defined decisions rejecting H₀',
       title='2,000 repetitions per population · n=250 · two-sided α=.05')
ax.plot([-.25,.25],[ALPHA,ALPHA],color='#327a9a',linestyle='--',label='Nominal α (null population only)')
for i,row in error_rate_sim.iterrows():
    if pd.notna(row.rejection_rate):
        ax.annotate(f'{row.rejection_rate:.3f} ± {row.MC95_margin:.3f}',
            (i,float(row.rejection_rate)), xytext=(0,13), textcoords='offset points',ha='center')
ax.legend(loc='center left',fontsize=9)
ax.grid(axis='y', alpha=.2)
fig.text(.5,.01,'Bars: approximate 95% Monte Carlo margins; consult the table for undefined-run counts.',ha='center',fontsize=9)
plt.tight_layout(rect=(0,.04,1,1))
plt.show()

### What to notice

Your summary has three rows, one per practice population, built from 6,000 repetition rows. Why would combining all 6,000 decisions into one rate mix different questions about errors and power?

Use `error_rate_sim` to identify the Type I-error rate, power and Type II-error rate. A and B differ from the reference by the same amount but have different spreads: how does this help explain their power? The **Monte Carlo margins** show that these rates are estimates from 2,000 repetitions. Module 9 takes up formal power planning.

Use these outputs for **L7-05 and L7-08**. Next, we will reveal the main-roster means and revisit your original decision for the remaining part of L7-07.


## Section 6B. What can we learn after revealing the population mean?

**Pause before scrolling further or running the next cell.** Keep the answer you reached in Section 5: your target population, estimated mean and interval, p-value and decision, and what the difference interval says about the ±0.5-day boundaries. Explain it as if the population mean were unknown. You do not need to submit an additional notebook response.

Our fictional study gives us an unusual opportunity: we have the full roster of people, so we can calculate each eligible population's exact mean, including documented zeros. In a real sample-based study, we usually would not know that answer.

The next cell reveals those roster means. They describe these fixed fictional populations, not national youth populations. Use the reveal to learn about your decision; keep it separate from the evidence available when you first made that decision.


In [ ]:
# Delayed truth reveal: this is the first calculation/display of main-roster means.
def classify_decision(null_is_true, reject):
    if null_is_true:
        return 'Type I error' if reject else 'correct nonrejection'
    return 'correct rejection' if reject else 'Type II error'

truth_rows = []
for branch, context, codes in [
    ('demo','Medical + Recreational',[1,2]),
    ('medical','Medical',[2]),
    ('recreational','Recreational',[1]),
]:
    values = roster.loc[
        roster[WINDOW_STATUS].eq('stable') & roster[POLICY].isin(codes)
        & roster[STATUS].isin(['observed','documented_nonuse']) & roster[OUTCOME].notna(), OUTCOME
    ].to_numpy(float)
    # Whole-number use-days make the null check exact: compare totals, not rounded means.
    assert np.equal(values, np.floor(values)).all()
    total_days = sum(int(value) for value in values)
    population_N = len(values)
    true_mean = total_days / population_N
    null_is_true = (total_days == MU0 * population_N)
    truth_rows.append(dict(branch=branch, context=context, eligible_N=population_N,
        total_use_days=total_days, true_mean=true_mean, true_difference=true_mean-MU0,
        null_is_true=null_is_true, true_departure_at_least_delta=abs(true_mean-MU0)>=DELTA))
roster_truth = pd.DataFrame(truth_rows)
display(roster_truth[['context','eligible_N','true_mean','true_difference','null_is_true','true_departure_at_least_delta']])
print('Mean/difference units:', UNITS, '| Meaningful-difference threshold:', DELTA)

decision_rows = []
for row in test_summary.to_dict('records'):
    branch = 'demo' if row['role']=='demo' else CHOICE
    truth = roster_truth.loc[roster_truth.branch.eq(branch)].iloc[0]
    decision_rows.append(dict(role=row['role'],context=row['context'],
        sample_mean=row['mean'],p_value=row['p_value'],decision=row['decision'],
        true_mean=float(truth.true_mean),true_difference=float(truth.true_difference),
        null_is_true=bool(truth.null_is_true),reject_H0=bool(row['reject_H0']),
        classification=classify_decision(bool(truth.null_is_true),bool(row['reject_H0'])),
        CI_covers_roster_mean=bool(row['lower']<=truth.true_mean<=row['upper']),
        true_departure_at_least_delta=bool(truth.true_departure_at_least_delta)))
decision_audit = pd.DataFrame(decision_rows)
display(decision_audit[['role','context','decision','classification','CI_covers_roster_mean']])

### What to notice after the reveal

Read your own decision row. Each eligible roster's true mean differs from exactly 1, so a
failure to reject is a **Type II error for that roster**. The true departures are also smaller
than the 0.5-day criterion: a mean can differ from exactly 1 while the difference stays below
the meaningful threshold. One classified decision cannot estimate an error rate.

**Recreational check:** your sample estimate fell below 1 while its revealed roster mean is
above 1. What does this reversal show about interpreting direction from one sample?

For **L7-07**, discuss what could follow from missing a real departure, and judge its importance
using the 0.5-day criterion. Then imagine a **different roster whose true mean is exactly 1**:
which decision would be a Type I error, and what unnecessary action might it prompt?


## Section 7. Bring the evidence together

We began by asking what a sample could tell us about a population mean relative to a 1-day reference. You now have an estimate and interval, a test decision, a judgment about possible magnitude, and examples of how decisions can be mistaken.

The supplied `results_record` gathers the sample-based numbers and settings for your **Canvas reflection**. The later population-truth reveal stays in the separate `decision_audit` table. Keeping the two apart helps you explain what the sample supported before you knew the full-roster answer. The `your_turn` row means your chosen-context analysis; it is not another coding task.


In [ ]:
# Supplied evidence record: numerical results and provenance, not a written conclusion.
results_record = test_summary[['role','context','eligible_N','n','mean','SE','z','p_value',
    'lower','upper','decision','difference','difference_lower','difference_upper','units']].copy()
results_record = results_record.merge(
    interpretation_table[['role','magnitude_relation']],on='role',how='left',validate='one_to_one')
results_record['dataset_version'] = VERSION
results_record['notebook_version'] = NOTEBOOK_VERSION
results_record['seed'] = results_record.role.map({'demo':DEMO_SEED,'your_turn':STUDENT_SEED})
results_record['reference_days'] = MU0
results_record['meaningful_difference_days'] = DELTA
results_record['alpha'] = ALPHA
results_record['method'] = 'two-sided standard normal; sample SD/sqrt(n); no FPC'
results_record['evidence_stage'] = 'sample-based; main-roster truth intentionally separate'
display(results_record[['role','context','n','mean','lower','upper','p_value','decision','magnitude_relation']])
print('Use the chosen-context row and retain units, dataset/version and seed. No extra notebook upload is required.')

### Prepare your synthesis

Use your earlier answers as evidence for **L7-10**; you may refer to **L7-07 and L7-09** instead
of copying their numbers. Brief, labeled parts are fine.

1. **The sample-based conclusion.** Name the eligible stable context, outcome and units; connect your decision and magnitude reasoning while treating the population mean as unknown.
2. **The descriptive finding, kept separate.** Cite one exploration pattern with its available-outcome denominator. Explain why neither it nor the fixed-reference test establishes a Medical–Recreational difference or a causal policy effect.
3. **A limitation and a safeguard.** For example, the policy link assumes unchanged residence. In analogous real data, combining visit dates, geography and sensitive outcomes can increase identification risk; controlled access and only the geographic detail needed reduce that risk without guaranteeing anonymity.

| Canvas evidence | Notebook source |
| --- | --- |
| L7-01–03: plan, eligibility and denominators | `analysis_plan`, `context_audit`, `exploration_summary` |
| L7-04: standardized statistic (z), p-value and decision | `null_model`, `test_summary`, null-tail display |
| L7-05, L7-08: error rates, power and simulation reasoning | `error_rate_sim`, simulation outputs |
| L7-07: sample evidence and the later truth check | `test_summary`, `decision_audit` |
| L7-09: magnitude | `interpretation_table` |
| L7-10: synthesis | `results_record`, exploration finding, limits and safeguards |
| L7-06: AI-use attestation | Course policy below |

### Independent Canvas reflection

Canvas responses must be your own; generative AI must not answer, draft, rewrite or check them.
Approved accessibility tools, spelling/grammar checkers and instructor-authorized translation
remain permitted. Complete the AI-use attestation accurately. No notebook submission is required.
